# Chapter 13 - Loading and Preprocessing Data with TensorFlow

## 1. The Data API Basics

**tf.data API** menyediakan abstraksi **`Dataset`** sebagai sebuah *sequence* item yang dapat dibaca dari memori maupun file, kemudian diolah secara efisien melalui berbagai transformasi.

Dataset bersifat **immutable**: setiap operasi (misalnya `map`, `batch`, atau `repeat`) tidak memodifikasi Dataset secara langsung, melainkan menghasilkan **Dataset baru**. Hal ini memungkinkan transformasi dikomposisikan secara fleksibel melalui *method chaining*.

---

### Core Dataset Operations

Beberapa operasi penting yang didukung oleh `tf.data.Dataset` meliputi:
- **`map`**: melakukan *preprocessing* pada setiap elemen
- **`filter`**: menyaring elemen berdasarkan kondisi tertentu
- **`batch` / `unbatch`**: mengelompokkan atau memecah data
- **`repeat`**: mengulang dataset untuk beberapa epoch
- **`shuffle`**: mengacak urutan data
- **`take`**: mengambil sejumlah elemen pertama
- **`interleave`**: membaca dari beberapa sumber secara paralel
- **`prefetch`**: melakukan *overlap* antara preprocessing di CPU dan training di GPU

Operasi-operasi ini biasanya dikombinasikan untuk membangun **pipeline training** yang bersifat i.i.d., ter-*shuffle*, dan efisien secara komputasi.


## **Example: Basic Dataset, Chaining, Map, Shuffle, Batch, Take**

In [1]:
import tensorflow as tf

# 1) Dataset sederhana dari tensor
X = tf.range(10)
dataset = tf.data.Dataset.from_tensor_slices(X)

for item in dataset:
    print(item)  # 0..9

# 2) Chaining: repeat, batch
dataset = dataset.repeat(3).batch(7)
for item in dataset:
    print(item)

# 3) Map: preprocessing per item
dataset = dataset.map(lambda x: x * 2)  # 0,2,4,...

# 4) Unbatch (experimental) + filter + take
dataset = dataset.apply(tf.data.experimental.unbatch())
dataset = dataset.filter(lambda x: x < 10)
for item in dataset.take(3):
    print(item)

# 5) Shuffle + batch
dataset = tf.data.Dataset.range(10).repeat(3)
dataset = dataset.shuffle(buffer_size=5, seed=42).batch(7)
for batch in dataset:
    print(batch)


Instructions for updating:
Use `tf.data.Dataset.unbatch()`.


tf.Tensor(0, shape=(), dtype=int32)
tf.Tensor(1, shape=(), dtype=int32)
tf.Tensor(2, shape=(), dtype=int32)
tf.Tensor(3, shape=(), dtype=int32)
tf.Tensor(4, shape=(), dtype=int32)
tf.Tensor(5, shape=(), dtype=int32)
tf.Tensor(6, shape=(), dtype=int32)
tf.Tensor(7, shape=(), dtype=int32)
tf.Tensor(8, shape=(), dtype=int32)
tf.Tensor(9, shape=(), dtype=int32)
tf.Tensor([0 1 2 3 4 5 6], shape=(7,), dtype=int32)
tf.Tensor([7 8 9 0 1 2 3], shape=(7,), dtype=int32)
tf.Tensor([4 5 6 7 8 9 0], shape=(7,), dtype=int32)
tf.Tensor([1 2 3 4 5 6 7], shape=(7,), dtype=int32)
tf.Tensor([8 9], shape=(2,), dtype=int32)
tf.Tensor(0, shape=(), dtype=int32)
tf.Tensor(2, shape=(), dtype=int32)
tf.Tensor(4, shape=(), dtype=int32)
tf.Tensor([0 2 3 6 7 9 4], shape=(7,), dtype=int64)
tf.Tensor([5 0 1 1 8 6 5], shape=(7,), dtype=int64)
tf.Tensor([4 8 7 1 2 3 0], shape=(7,), dtype=int64)
tf.Tensor([5 4 2 7 8 9 9], shape=(7,), dtype=int64)
tf.Tensor([3 6], shape=(2,), dtype=int64)


## 2. CSV Input Pipelines with tf.data

Untuk **data tabular berukuran besar** (misalnya dataset *California Housing*), pipeline input yang efisien biasanya mengikuti urutan berikut:

1. `list_files`
2. `interleave` beberapa `TextLineDataset` (dengan melewati header)
3. `map(preprocess)`
4. `shuffle`
5. `repeat` (opsional)
6. `batch`
7. `prefetch`

Pendekatan ini memungkinkan:
- pembacaan paralel dari banyak file,
- proses *shuffling* yang baik,
- *overlap* antara pemuatan/preprocessing data di CPU dan pelatihan di GPU,

sehingga **bottleneck I/O** dapat dikurangi secara signifikan.

---

### Preprocessing CSV Data

Fungsi *preprocessing* umumnya menggunakan:
- **`tf.io.decode_csv`** untuk mem-parsing setiap baris CSV menjadi fitur dan label,
- serta operasi **scaling** (normalisasi atau standardisasi) langsung di dalam pipeline `tf.data`.

Dengan cara ini, seluruh alur input data tetap berada di dalam graph TensorFlow dan dapat dioptimalkan secara end-to-end.

## **Example: CSV Reader Dataset + Preprocess + Keras**

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

# Precomputed mean & std per feature (misal dari NumPy)
X_mean = tf.constant([...], dtype=tf.float32)
X_std  = tf.constant([...], dtype=tf.float32)
n_inputs = 8

def preprocess(line):
    # record_defaults: 8 feature float (default 0.), 1 target float (no default)
    defs = [0.] * n_inputs + [tf.constant((), dtype=tf.float32)]
    fields = tf.io.decode_csv(line, record_defaults=defs)
    x = tf.stack(fields[:-1])
    y = tf.stack(fields[-1:])
    x = (x - X_mean) / X_std
    return x, y

def csv_reader_dataset(filepaths, repeat=1, n_readers=5,
                       n_read_threads=None, shuffle_buffer_size=10000,
                       n_parse_threads=5, batch_size=32):
    dataset = tf.data.Dataset.list_files(filepaths)
    dataset = dataset.interleave(
        lambda fp: tf.data.TextLineDataset(fp).skip(1),
        cycle_length=n_readers,
        num_parallel_calls=n_read_threads
    )
    dataset = dataset.map(preprocess,
                          num_parallel_calls=n_parse_threads)
    dataset = dataset.shuffle(shuffle_buffer_size).repeat(repeat)
    dataset = dataset.batch(batch_size).prefetch(1)
    return dataset

train_set = csv_reader_dataset(train_filepaths)
valid_set = csv_reader_dataset(valid_filepaths)
test_set  = csv_reader_dataset(test_filepaths)

model = keras.models.Sequential([
    keras.layers.Dense(30, activation="elu",
                       kernel_initializer="he_normal"),
    keras.layers.Dense(1)
])
model.compile(loss="mse", optimizer="nadam")
model.fit(train_set, epochs=10, validation_data=valid_set)
model.evaluate(test_set)


## 3. TFRecord Format & Protocol Buffers

**TFRecord** adalah format file biner sederhana yang berisi *sequence* record dengan panjang variabel. Format ini sangat cocok untuk **data berukuran besar dan kompleks** (misalnya gambar, audio, atau tensor arbitrer) karena:
- lebih cepat dibaca dibanding format teks seperti CSV,
- lebih hemat ruang penyimpanan,
- dan mudah diparalelkan dalam pipeline input.

Umumnya, setiap record TFRecord berisi objek **Protocol Buffers** khusus TensorFlow, yaitu:
- **`Example`** untuk data satu contoh (fixed-length),
- **`SequenceExample`** untuk data sekuensial (misalnya time series atau teks).

Parsing dilakukan sepenuhnya di dalam TensorFlow menggunakan operasi seperti:
- `tf.io.parse_single_example`
- `tf.io.parse_single_sequence_example`

---

### Typical TFRecord Pipeline

Pipeline penggunaan TFRecord biasanya terdiri dari dua tahap:
1. **Konversi offline** (misalnya dari CSV atau image folder ke TFRecord berisi `Example` / `SequenceExample`)
2. **Training-time pipeline**:
   - `TFRecordDataset`
   - parsing protobuf
   - preprocessing standar (`map`, `batch`, `prefetch`)

Pendekatan ini memisahkan preprocessing berat dari training loop dan meningkatkan performa I/O secara signifikan.


## **Example: Basic TFRecord Write & Read**

In [2]:
import tensorflow as tf

# Write raw strings to TFRecord
with tf.io.TFRecordWriter("my_data.tfrecord") as f:
    f.write(b"This is the first record")
    f.write(b"And this is the second record")

# Read TFRecord
dataset = tf.data.TFRecordDataset(["my_data.tfrecord"])
for item in dataset:
    print(item)  # scalar string tensors


tf.Tensor(b'This is the first record', shape=(), dtype=string)
tf.Tensor(b'And this is the second record', shape=(), dtype=string)


## **Example: tf.train.Example & Parsing**

In [ ]:
from tensorflow.train import BytesList, FloatList, Int64List
from tensorflow.train import Feature, Features, Example

# Build Example
person_example = Example(
    features=Features(
        feature={
            "name":   Feature(bytes_list=BytesList(value=[b"Alice"])),
            "id":     Feature(int64_list=Int64List(value=[123])),
            "emails": Feature(bytes_list=BytesList(
                value=[b"a@b.com", b"c@d.com"]
            )),
        }
    )
)

with tf.io.TFRecordWriter("my_contacts.tfrecord") as f:
    f.write(person_example.SerializeToString())

# Parse with tf.io.parse_single_example
feature_description = {
    "name":   tf.io.FixedLenFeature([], tf.string, default_value=""),
    "id":     tf.io.FixedLenFeature([], tf.int64, default_value=0),
    "emails": tf.io.VarLenFeature(tf.string),
}

for serialized in tf.data.TFRecordDataset(["my_contacts.tfrecord"]):
    parsed = tf.io.parse_single_example(serialized, feature_description)
    name   = parsed["name"]
    pid    = parsed["id"]
    emails = parsed["emails"].values  # sparse → dense values


## 4. Preprocessing Numerical & Categorical Features

Preprocessing fitur numerik dan kategorikal dapat dilakukan di beberapa level, tergantung kebutuhan dan target deployment:

1. **Di luar TensorFlow**  
   Menggunakan NumPy, pandas, atau scikit-learn (misalnya `StandardScaler`, `OneHotEncoder`).

2. **Di dalam pipeline `tf.data`**  
   Menggunakan `map(preprocess_fn)` agar preprocessing berjalan paralel dan terintegrasi dengan training.

3. **Sebagai preprocessing layer di dalam model Keras**  
   Pendekatan ini sangat dianjurkan untuk *production*, karena preprocessing ikut di-*export* bersama model.

---

### Examples

- **Standardization layer** kustom dengan metode `adapt`, mirip dengan `StandardScaler`
- **Categorical encoding** menggunakan:
  - lookup table (`StringLookup` / `IntegerLookup`)
  - one-hot encoding
  - atau embedding untuk kategori berukuran besar

Pendekatan berbasis layer memastikan konsistensi antara training dan inference.

## **Example: Standardization Layer**

In [ ]:
import numpy as np
from tensorflow import keras

class Standardization(keras.layers.Layer):
    def adapt(self, data_sample):
        self.means_ = np.mean(data_sample, axis=0, keepdims=True)
        self.stds_  = np.std(data_sample, axis=0, keepdims=True)

    def call(self, inputs):
        eps = keras.backend.epsilon()
        return (inputs - self.means_) / (self.stds_ + eps)

# Usage
std_layer = Standardization()
std_layer.adapt(X_train_sample)

model = keras.models.Sequential([
    std_layer,
    keras.layers.Dense(30, activation="relu"),
    keras.layers.Dense(1)
])


## **Example: Categorical → One-Hot with Lookup Table**

In [4]:
import tensorflow as tf

vocab = ["<1H OCEAN", "INLAND", "NEAR OCEAN", "NEAR BAY", "ISLAND"]
indices = tf.range(len(vocab), dtype=tf.int64)
table_init = tf.lookup.KeyValueTensorInitializer(vocab, indices)

num_oov_buckets = 2
table = tf.lookup.StaticVocabularyTable(table_init, num_oov_buckets)

categories = tf.constant(["NEAR BAY", "DESERT", "INLAND", "INLAND"])
cat_indices = table.lookup(categories)

cat_one_hot = tf.one_hot(
    cat_indices,
    depth=len(vocab) + num_oov_buckets
)


## **Example: Categorical → Embedding (Manual & Keras Layer)**

In [5]:
embedding_dim = 2
embed_init = tf.random.uniform(
    [len(vocab) + num_oov_buckets, embedding_dim]
)
embedding_matrix = tf.Variable(embed_init)

categories = tf.constant(["NEAR BAY", "DESERT", "INLAND", "INLAND"])
cat_indices = table.lookup(categories)

# Manual lookup
embedded = tf.nn.embedding_lookup(embedding_matrix, cat_indices)

# Using Keras Embedding layer
from tensorflow import keras
embedding = keras.layers.Embedding(
    input_dim=len(vocab) + num_oov_buckets,
    output_dim=embedding_dim
)
embedded2 = embedding(cat_indices)


## **Example: Combine Numeric + Categorical Embedding in Model**

In [ ]:
from tensorflow import keras

regular_inputs = keras.layers.Input(shape=[8])
categories     = keras.layers.Input(shape=[], dtype=tf.string)

cat_indices = keras.layers.Lambda(
    lambda cats: table.lookup(cats)
)(categories)

cat_embed = keras.layers.Embedding(input_dim=6, output_dim=2)(cat_indices)

encoded = keras.layers.concatenate([regular_inputs, cat_embed])
outputs = keras.layers.Dense(1)(encoded)

model = keras.models.Model(
    inputs=[regular_inputs, categories],
    outputs=[outputs]
)


## 5. TF Transform & Keras Preprocessing Layers

**TensorFlow Transform (TF Transform / TFT)** adalah bagian dari **TFX** yang memungkinkan preprocessing didefinisikan **satu kali** dan digunakan secara konsisten untuk training dan serving.

TF Transform bekerja dengan cara:
- menjalankan preprocessing secara **batch** pada seluruh training set menggunakan **Apache Beam**,
- lalu mengompilasi preprocessing tersebut menjadi **TF Function** yang dapat ditanam langsung ke model saat serving.

Keunggulan utama pendekatan ini adalah **menghindari train/serving skew**, karena preprocessing yang sama persis digunakan di kedua fase.

---

### Keras Preprocessing Layers

Sebagai alternatif yang lebih ringan, Keras menyediakan **preprocessing layers bawaan**, antara lain:
- `Normalization`
- `Discretization`
- `TextVectorization`
- `StringLookup` / `IntegerLookup`

Pola penggunaannya umumnya:
1. Membuat layer preprocessing
2. Memanggil `adapt(data_sample)`
3. Menggunakannya sebagai **layer pertama** dalam model

Pendekatan ini mudah diintegrasikan dan cocok untuk sebagian besar use case non-TFX.


## **Example: TF Transform Preprocess Function (Sketch)**

In [ ]:
import tensorflow_transform as tft

def preprocess(inputs):
    median_age      = inputs["housing_median_age"]
    ocean_proximity = inputs["ocean_proximity"]

    standardized_age = tft.scale_to_z_score(median_age)
    ocean_id = tft.compute_and_apply_vocabulary(ocean_proximity)

    return {
        "standardized_median_age": standardized_age,
        "ocean_proximity_id": ocean_id,
    }


## **Example: Keras Normalization + Discretization Pipeline (Sketch)**


In [ ]:
from tensorflow import keras

normalization  = keras.layers.Normalization()
discretization = keras.layers.Discretization([...])

pipeline = keras.layers.PreprocessingStage([normalization, discretization])
pipeline.adapt(data_sample)  # compute stats, bins, etc.

model = keras.models.Sequential([
    pipeline,
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(1)
])


## 6. TensorFlow Datasets (TFDS)

**TensorFlow Datasets (TFDS)** menyediakan akses mudah ke berbagai dataset populer, seperti:
- MNIST, Fashion-MNIST
- CIFAR
- ImageNet
- berbagai dataset NLP dan speech

TFDS secara otomatis:
- mendownload dataset,
- memverifikasi checksum,
- dan menyajikannya sebagai **`tf.data.Dataset`** siap pakai.

---

### TFDS Usage Patterns

`tfds.load()` dapat mengembalikan:
- dictionary dataset (misalnya `{"train": ds_train, "test": ds_test}`), atau
- pasangan `(features, labels)` jika `as_supervised=True`.

Pipeline yang umum digunakan:
1. `tfds.load`
2. `shuffle`
3. `batch`
4. `prefetch`
5. langsung digunakan pada `model.fit()`

TFDS sangat membantu untuk **eksperimen cepat**, benchmarking, dan pembelajaran tanpa harus menulis pipeline input dari nol.

## **Example: MNIST via TFDS + Keras**

In [ ]:
import tensorflow_datasets as tfds
from tensorflow import keras

dataset = tfds.load(
    name="mnist",
    batch_size=32,
    as_supervised=True
)
mnist_train, mnist_test = dataset["train"], dataset["test"]

mnist_train = mnist_train.prefetch(1)

model = keras.models.Sequential([
    keras.layers.Reshape([28, 28, 1], input_shape=[28, 28, 1]),
    keras.layers.Flatten(),
    keras.layers.Dense(300, activation="relu"),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])

model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

model.fit(mnist_train, epochs=5)
